# Chapter 4: Discrete Behavior Cloning

This Colab follows the Chapter 4 manuscript from the multimodal regression trap through a trained, discrete SO-101 action policy. A GPU runtime is strongly recommended for the real backbone cells.

In [ ]:
# Colab setup: Chapter 4 builds on the live Chapter 2 and 3 packages.
import os, sys, subprocess, time, shutil, base64
from pathlib import Path

if 'google.colab' in sys.modules:
    repos = [
        'lrm-code-chapter-2',
        'lrm-code-chapter-3',
        'lrm-code-chapter-4',
    ]
    base = 'https://github.com/Large-Robotics-Models-From-Scratch'
    revisions = {'lrm-code-chapter-4': 'gpt_prototype'}
    # Optional: add a fine-grained PAT as a Colab Secret named GITHUB_TOKEN.
    # It needs read access to each private chapter repository.
    from google.colab import userdata
    try:
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        token = None
    git_env = os.environ.copy()
    if token:
        credentials = base64.b64encode(
            f'x-access-token:{token}'.encode()).decode()
        git_env.update({
            'GIT_CONFIG_COUNT': '1',
            'GIT_CONFIG_KEY_0': 'http.extraHeader',
            'GIT_CONFIG_VALUE_0': f'Authorization: Basic {credentials}',
        })
    for repo in repos:
        path = Path('/content') / repo
        if (path / '.git').is_dir():
            print(f'Using existing checkout: {path}')
            continue
        if path.exists():
            raise RuntimeError(
                f'{path} exists but is not a complete Git checkout. '
                'Choose Runtime > Disconnect and delete runtime, then rerun this cell.')
        url = f'{base}/{repo}.git'
        staging = Path('/content') / f'.{repo}.clone'
        last_error = ''
        for attempt in range(1, 4):
            if staging.exists():
                shutil.rmtree(staging)
            revision = revisions.get(repo, 'main')
            result = subprocess.run(
                ['git', 'clone', '--depth', '1', '--branch', revision,
                 url, str(staging)],
                text=True, capture_output=True, env=git_env)
            if result.returncode == 0:
                staging.rename(path)
                print(f'Cloned {repo}')
                break
            last_error = result.stderr.strip()
            print(f'Clone attempt {attempt}/3 failed for {repo}:\n{last_error}')
            time.sleep(2 ** attempt)
        else:
            hint = (
                ' Add a Colab Secret named GITHUB_TOKEN with read access '
                'to the private repository, enable notebook access, then rerun.'
                if not token else
                ' Check that GITHUB_TOKEN can read this repository and that '
                'organization SSO is authorized, then rerun.')
            raise RuntimeError(
                f'Could not clone {url}:\n{last_error}\n{hint}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '-e', '/content/lrm-code-chapter-2[data]',
                    '-e', '/content/lrm-code-chapter-3',
                    '-e', '/content/lrm-code-chapter-4[data]'], check=True)

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

from ch04.constants import ACTION_BINS, ACTION_DIM, ACTION_HORIZON

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print(f'label grid: H={ACTION_HORIZON}, D={ACTION_DIM}, B={ACTION_BINS}')

## 4.2 The multimodal regression trap

The observation contains no clue about which of two equally valid expert modes was chosen. MSE therefore learns the conditional mean: zero, where the demonstrations have almost no density.

In [ ]:
from ch04.exercises import make_bimodal_actions, train_mse_baseline

observations, expert_actions = make_bimodal_actions()
mse_model, mse_history = train_mse_baseline()
with torch.no_grad():
    collapsed = mse_model(torch.zeros(1, 1)).item()
print(f'MSE prediction: {collapsed:+.3f} (expert modes are -1 and +1)')

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(expert_actions.numpy(), bins=40)
axes[0].axvline(collapsed, color='crimson', label='MSE prediction')
axes[0].legend(); axes[0].set_xlabel('action')
axes[1].plot(mse_history); axes[1].set(xlabel='step', ylabel='MSE')
plt.show()

## 4.3 Fit the action tokenizer

Chapter 2's loader already z-score normalizes its frame-wise action column. We fit per-joint q01/q99 limits in exactly that normalized space. The Colab default samples 64 batches so this early step does not decode the entire video dataset; set `TOKENIZER_FIT_BATCHES = None` only when you want the full-data fit.

In [ ]:
from ch04 import fit_action_tokenizer
from ch04.data import (DEFAULT_DATASET_ID, collect_normalized_actions,
                         make_chunked_dataloaders)

train_loader, validation_loader, stats = make_chunked_dataloaders(
    DEFAULT_DATASET_ID, batch_size=4, validation_fraction=0.1, seed=7)
# This samples 64 video-decoded batches (~4k chunked actions). Set to None
# for the full-data percentile fit after confirming the pipeline works.
TOKENIZER_FIT_BATCHES = 64
normalized_actions = collect_normalized_actions(
    train_loader, stats, max_batches=TOKENIZER_FIT_BATCHES)
tokenizer = fit_action_tokenizer(normalized_actions)

example = normalized_actions[0]
bins = tokenizer.encode(example)
tokens = tokenizer.bins_to_tokens(bins)
decoded = tokenizer.decode(bins)
print('bins:   ', bins)
print('LM ids: ', tokens, '(native tail; no vocabulary growth)')
print('max normalized round-trip error:', np.abs(decoded-example).max())
print('tokenizer fit batches:', TOKENIZER_FIT_BATCHES or 'all')
print('train episodes:', train_loader.dataset.episodes)
print('validation episodes:', validation_loader.dataset.episodes)

In [ ]:
# One action chunk becomes a timestep-major sequence of 96 labels.
demo_grid = torch.arange(ACTION_HORIZON * ACTION_DIM).reshape(
    1, ACTION_HORIZON, ACTION_DIM)
demo_flat = demo_grid.flatten(1)
print('grid:', tuple(demo_grid.shape), '-> tokens:', tuple(demo_flat.shape))
print('first two timesteps:', demo_flat[0, :12].tolist())

## 4.4 Build the three action heads

The factorized head is the one-shot baseline, the autoregressive head is the exact chain-rule model, and the bidirectional parallel head is the manuscript's one-pass training and evaluation path.

In [ ]:
from ch03 import VLABackbone
from ch04 import (AutoregressiveActionHead, FactorizedActionHead,
                  ParallelDecodeActionHead)

backbone = VLABackbone().to(device)
factorized = FactorizedActionHead().to(device)
autoregressive = AutoregressiveActionHead(backbone).to(device)
head = ParallelDecodeActionHead(backbone).to(device)
print('factorized grid:', factorized.grid)
print('autoregressive reserved base:', autoregressive.token_base)
print('parallel grid:', head.grid)

## 4.5 Load a real action chunk and run one forward pass

In [ ]:
from ch04.data import prepare_batch, action_targets
from ch04.backbone_adapter import encode_prefix, gather_state_hidden
from ch04.losses import masked_token_cross_entropy

batch = next(iter(validation_loader))
model_inputs = prepare_batch(batch, stats, backbone, device)
target_bins, token_pad = action_targets(
    batch, stats, tokenizer, device)

with torch.no_grad():
    prefix_hidden = encode_prefix(backbone, *model_inputs)
    state_hidden = gather_state_hidden(
        backbone, prefix_hidden, model_inputs[1])
    factorized_logits = factorized(state_hidden)
    parallel_logits = head(*model_inputs)
    ar_logits = autoregressive.teacher_forced_logits(
        *model_inputs, target_bins)
    loss = masked_token_cross_entropy(
        parallel_logits, target_bins, token_pad)
print('actions:', tuple(batch['action'].shape))
print('factorized:', tuple(factorized_logits.shape))
print('autoregressive:', tuple(ar_logits.shape))
print('parallel:', tuple(parallel_logits.shape))
print(f'initial CE={loss.item():.3f}; log(256)={math.log(256):.3f}')

In [ ]:
# Inference is sequential for AR; decode one six-joint timestep here.
with torch.no_grad():
    ar_first_timestep = autoregressive.decode(
        *model_inputs, grid_size=ACTION_DIM, greedy=True)
print('AR sequential bins:', ar_first_timestep[0].tolist())

## 4.5.3 Train the policy

Ten steps are a pipeline check. Set `TRAIN_STEPS = 20_000` for the manuscript run and save the Colab runtime between sessions.

In [ ]:
from ch04.train import train_action_head

TRAIN_STEPS = 10  # change to 20_000 for the full experiment
COMPARISON_STEPS = 2  # short baseline checks; increase for fair comparison
histories = {}
for name, action_head, steps in [
    ('factorized', factorized, COMPARISON_STEPS),
    ('autoregressive', autoregressive, COMPARISON_STEPS),
    ('parallel', head, TRAIN_STEPS),
]:
    histories[name] = train_action_head(
        action_head, backbone, train_loader, stats, tokenizer, device,
        total_steps=steps,
        warmup_steps=min(5, steps - 1),
        log_every=1,
        checkpoint_dir=f'/content/ch04-checkpoints/{name}')
history = histories['parallel']

## 4.6 Inspect the learned distribution

In [ ]:
from ch04.decoding import evaluation_mode, sample_logits
from ch04.diagnostics import (joint_mismatch_rate, plot_action_distribution,
                                plot_joint_pairs, sample_cell_pairs,
                                temporal_jitter)

with torch.no_grad(), evaluation_mode(head):
    parallel_logits = head(*model_inputs)
with torch.no_grad(), evaluation_mode(factorized):
    prefix_hidden = encode_prefix(backbone, *model_inputs)
    state_hidden = gather_state_hidden(
        backbone, prefix_hidden, model_inputs[1])
    factorized_logits = factorized(state_hidden)
with torch.no_grad(), evaluation_mode(autoregressive):
    ar_logits = autoregressive.teacher_forced_logits(
        *model_inputs, target_bins)
probabilities = parallel_logits[0, 0].softmax(-1).cpu().numpy()
plot_action_distribution(probabilities)
plt.title('Policy distribution: timestep 0, joint 0')
plt.show()

# Compare gripper/wrist marginal samples at the first timestep.
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for axis, (name, values) in zip(axes, [
    ('factorized', factorized_logits),
    ('autoregressive (teacher forced)', ar_logits),
    ('parallel', parallel_logits),
]):
    wrist, gripper = sample_cell_pairs(values, 4, 5)
    plot_joint_pairs(wrist, gripper, ax=axis, title=name)
    print(name, 'mismatch:', joint_mismatch_rate(
        wrist, gripper, np.median(wrist), np.median(gripper)))
plt.tight_layout(); plt.show()

for name, values in [('factorized', factorized_logits),
                     ('autoregressive', ar_logits),
                     ('parallel', parallel_logits)]:
    sampled = sample_logits(values, greedy=False)[0].reshape(
        ACTION_HORIZON, ACTION_DIM).cpu().numpy()
    print(name, 'temporal jitter:', temporal_jitter(sampled))

## 4.7 Decode tokens back to controls

The tokenizer returns normalized actions. The decoder then applies Chapter 2's inverse statistics so predictions and expert actions are compared in the same raw dataset units. This is open-loop validation, not a claim of safe robot deployment.

In [ ]:
from ch04.decoding import (decode_parallel_chunk,
                             mean_absolute_error_by_timestep)
from ch04.diagnostics import plot_chunk_comparison

prediction = decode_parallel_chunk(
    head, model_inputs, tokenizer, stats, top_p=0.95)
expert = batch['action'].float()
pad = batch.get('action_is_pad')
if pad is not None:
    pad = pad.bool()
mae = mean_absolute_error_by_timestep(prediction, expert, pad)
print('MAE by chunk timestep:', mae.numpy().round(4))
plot_chunk_comparison(prediction[0].numpy(), expert[0].numpy())
plt.show()

## Next experiments

- Train all three heads for equal step counts before interpreting their diagnostic differences.
- Train the main parallel head for 20k steps and compare checkpoint entropy.
- Compare chunk-by-chunk execution with `TemporalEnsembler` on validation episodes.
- Keep physical deployment separate: dataset units and the simulator/robot control mode must be converted and safety-checked explicitly.